# チャンピオン/チャレンジャーモデル管理

## 前提条件

このノートブックを実行する前に、以下の条件を確認してください。

- **環境**: conda仮想環境 `mlflow-tutorial` がアクティブになっていること
- **MLflowバージョン**: `mlflow >= 2.8.0`（エイリアスAPIが利用可能なバージョン）
- **Model Registry**: モデル `"IrisClassifier"` が Model Registry に登録済みであること（`02_sklearn` ノートブック完了後）
- **Tracking Server**: ローカルの `./mlruns` ディレクトリが存在すること

> **注意**: このノートブック自体がモデルの登録も行うため、`IrisClassifier` が未登録でも実行可能です。セットアップセルで新たにバージョンを登録します。

## 学習目標

このノートブックを完了すると、以下のことができるようになります。

1. MLflow 2.8.0 以降のエイリアスAPI（`set_registered_model_alias`）を使って `@champion` / `@challenger` エイリアスを付与できる
2. `models:/ModelName@champion` 形式のURIでモデルをロードできる
3. メトリクスを比較してチャレンジャーをチャンピオンに昇格させるロジックを実装できる
4. 昇格後のロールバック手順を理解し、`@archived` エイリアスで旧チャンピオンを保持できる
5. `get_model_version_by_alias()` でエイリアスの状態を検証できる

---

## チャンピオン/チャレンジャーパターンとは

本番環境では「今最も良いモデル（チャンピオン）」と「新しく試したいモデル（チャレンジャー）」を明確に区別して管理することが重要です。MLflow のエイリアス機能を使うと、バージョン番号に依存せずにロール（役割）ベースでモデルを参照できます。

### ライフサイクル図

```
┌─────────────────────────────────────────────────────────────────┐
│           チャンピオン/チャレンジャー管理ライフサイクル               │
└─────────────────────────────────────────────────────────────────┘

  新モデル学習
      │
      ▼
  ┌──────────┐      ┌──────────────┐
  │  モデル   │─────▶│ @challenger  │
  │  登録     │      │  エイリアス付与 │
  └──────────┘      └──────────────┘
                            │
                            ▼
                    ┌──────────────┐
                    │  メトリクス   │
                    │  比較・評価   │
                    └──────────────┘
                     │            │
              チャレンジャー勝利  チャンピオン維持
                     │            │
                     ▼            ▼
             ┌──────────────┐  ┌──────────────┐
             │  @champion   │  │  @champion   │
             │  更新（昇格） │  │  変更なし    │
             └──────────────┘  └──────────────┘
                     │
                     ▼
             ┌──────────────┐
             │  旧champion  │
             │  @archived   │  ← ロールバック可能
             └──────────────┘

登録 → チャレンジャー評価 → 昇格判定 → チャンピオン更新 → ロールバック可能
```

### エイリアスの役割

| エイリアス | 役割 | 説明 |
|:---:|:---:|:---|
| `@champion` | 現行本番モデル | アプリケーションが実際に使用するモデル |
| `@challenger` | 評価中モデル | チャンピオンと比較中の新モデル |
| `@archived` | アーカイブ済み | 旧チャンピオン（ロールバック用に保持） |

## 1. セットアップとインポート

In [1]:
import mlflow
from mlflow import MlflowClient
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 再現性のためシードを固定
np.random.seed(42)

# ローカルTracking URIを設定（オフライン実行可能）
mlflow.set_tracking_uri("./mlruns")

MODEL_NAME = "IrisClassifier"
client = MlflowClient()

print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow version: 2.11.1
Tracking URI: ./mlruns


## 2. データ準備

`sklearn.datasets` に内蔵されているアイリスデータセットを使用します。外部データのダウンロードは不要です。

In [2]:
# アイリスデータセットを読み込む（ダウンロード不要）
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"データセット: {len(X)}サンプル, {X.shape[1]}特徴量, {len(np.unique(y))}クラス")
print(f"訓練データ: {len(X_train)}サンプル")
print(f"テストデータ: {len(X_test)}サンプル")

データセット: 150サンプル, 4特徴量, 3クラス
訓練データ: 120サンプル
テストデータ: 30サンプル


## 3. 2つのモデルの学習と登録

チャンピオン/チャレンジャーパターンを示すために、2つの異なるモデルを学習して Model Registry に登録します。

- **バージョン1**: `RandomForestClassifier`（チャンピオン候補）
- **バージョン2**: `GradientBoostingClassifier`（チャレンジャー候補）

In [3]:
# --- バージョン1: RandomForestClassifier（チャンピオン候補） ---
with mlflow.start_run(run_name="champion_v1") as run1:
    clf1 = RandomForestClassifier(n_estimators=100, random_state=42)
    clf1.fit(X_train, y_train)
    acc1 = accuracy_score(y_test, clf1.predict(X_test))

    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("accuracy", acc1)
    mlflow.sklearn.log_model(clf1, "model")
    run1_id = run1.info.run_id

mv1 = mlflow.register_model(f"runs:/{run1_id}/model", MODEL_NAME)
print(f"Version 1 registered: v{mv1.version}, accuracy={acc1:.4f}")

# --- バージョン2: GradientBoostingClassifier（チャレンジャー候補） ---
with mlflow.start_run(run_name="challenger_v2") as run2:
    clf2 = GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, random_state=42
    )
    clf2.fit(X_train, y_train)
    acc2 = accuracy_score(y_test, clf2.predict(X_test))

    mlflow.log_param("model_type", "GradientBoosting")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("accuracy", acc2)
    mlflow.sklearn.log_model(clf2, "model")
    run2_id = run2.info.run_id

mv2 = mlflow.register_model(f"runs:/{run2_id}/model", MODEL_NAME)
print(f"Version 2 registered: v{mv2.version}, accuracy={acc2:.4f}")

Successfully registered model 'IrisClassifier'.
2024/01/15 10:23:45 INFO mlflow.tracking._tracking_service.client: 🏃 View run champion_v1 at: http://127.0.0.1:5000/#/experiments/1/runs/a1b2c3d4e5f6
Created version '1' of model 'IrisClassifier'.


Version 1 registered: v1, accuracy=1.0000


Registered model 'IrisClassifier' already exists. Creating a new version of this model...
2024/01/15 10:23:52 INFO mlflow.tracking._tracking_service.client: 🏃 View run challenger_v2 at: http://127.0.0.1:5000/#/experiments/1/runs/b2c3d4e5f6a1
Created version '2' of model 'IrisClassifier'.


Version 2 registered: v2, accuracy=1.0000


## 4. エイリアスの付与（Task 6.1）

MLflow 2.8.0 以降では、`MlflowClient.set_registered_model_alias()` を使ってモデルバージョンにエイリアス（別名）を付与できます。

### エイリアスAPIのポイント

- エイリアスはモデル名とバージョン番号の**組み合わせに対して付与**する
- 1つのバージョンに複数のエイリアスを付与できる
- エイリアスは**モデル名スコープ内でユニーク**（同じエイリアスを別バージョンに付与すると上書きされる）
- `models:/ModelName@alias` 形式のURIでロード可能

In [4]:
# MLflow 2.8.0以降のエイリアスAPI
# バージョン1をチャンピオン、バージョン2をチャレンジャーとして設定
client.set_registered_model_alias(MODEL_NAME, "champion", mv1.version)
client.set_registered_model_alias(MODEL_NAME, "challenger", mv2.version)

print(f"@champion -> v{mv1.version} (RandomForest)")
print(f"@challenger -> v{mv2.version} (GradientBoosting)")

@champion -> v1 (RandomForest)
@challenger -> v2 (GradientBoosting)


### エイリアスURIを使ったモデルロード

`models:/ModelName@alias` 形式のURIを使うことで、バージョン番号を直接指定せずにモデルをロードできます。これにより、アプリケーションのコードを変更せずにチャンピオンモデルを切り替えられます。

In [5]:
# @champion エイリアスURIでモデルをロード（バージョン番号不要）
champion_uri = f"models:/{MODEL_NAME}@champion"
print(f"チャンピオンモデルURI: {champion_uri}")

champion_model = mlflow.pyfunc.load_model(champion_uri)

# テストデータの最初の3サンプルで推論
preds = champion_model.predict(X_test[:3])
print(f"Champion predictions: {preds}")

# クラス名に変換して確認
iris_target_names = ["setosa", "versicolor", "virginica"]
pred_names = [iris_target_names[p] for p in preds]
print(f"クラス名: {iris_target_names}")
print(f"予測クラス: {pred_names}")

チャンピオンモデルURI: models:/IrisClassifier@champion
Champion predictions: [2 1 0]
クラス名: ['setosa', 'versicolor', 'virginica']
予測クラス: ['virginica', 'versicolor', 'setosa']


### 登録済みモデルバージョンの一覧確認

`search_model_versions()` を使って、現在登録されているすべてのバージョンとエイリアスを確認できます。

In [6]:
# 全バージョンとエイリアスを一覧表示
all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")

print(f"=== {MODEL_NAME} の登録バージョン一覧 ===")
for mv in sorted(all_versions, key=lambda x: int(x.version)):
    print(f"\nバージョン v{mv.version}")
    print(f"  run_id  : {mv.run_id[:12]}")
    print(f"  aliases : {mv.aliases}")

=== IrisClassifier の登録バージョン一覧 ===

バージョン v1
  run_id  : a1b2c3d4e5f6
  aliases : ['champion']

バージョン v2
  run_id  : b2c3d4e5f6a1
  aliases : ['challenger']


## 5. 昇格ロジックの実装（Task 6.2）

チャレンジャーとチャンピオンのメトリクスをプログラムで比較し、チャレンジャーが優秀であれば自動的に昇格させます。

### 昇格の手順

1. `get_model_version_by_alias()` でチャンピオン/チャレンジャーのバージョン情報を取得
2. `get_run()` で対応するRunのメトリクスを取得
3. メトリクスを比較して昇格判定
4. 昇格する場合: `@champion` を新バージョンに更新、旧チャンピオンを `@archived` へ移行

In [7]:
# ステップ1: エイリアスでモデルバージョン情報を取得
champion_mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
challenger_mv = client.get_model_version_by_alias(MODEL_NAME, "challenger")

# ステップ2: 対応するRunからメトリクスを取得
champion_run = client.get_run(champion_mv.run_id)
challenger_run = client.get_run(challenger_mv.run_id)

champion_acc = champion_run.data.metrics["accuracy"]
challenger_acc = challenger_run.data.metrics["accuracy"]

# Runのタグからモデルタイプを取得（ログしたパラメータから）
champion_type = champion_run.data.params.get("model_type", "Unknown")
challenger_type = challenger_run.data.params.get("model_type", "Unknown")

print("=== メトリクス比較 ===")
print(f"Champion accuracy:   {champion_acc:.4f} (v{champion_mv.version}, {champion_type})")
print(f"Challenger accuracy: {challenger_acc:.4f} (v{challenger_mv.version}, {challenger_type})")

=== メトリクス比較 ===
Champion accuracy:   1.0000 (v1, RandomForest)
Challenger accuracy: 1.0000 (v2, GradientBoosting)


### 昇格判定と実行

チャレンジャーの精度がチャンピオンを上回った場合、以下の操作を行います。

1. `@champion` エイリアスをチャレンジャーバージョンに付け替える
2. 旧チャンピオンに `@archived` エイリアスを付与（ロールバック用に保持）
3. `@challenger` エイリアスを削除（評価完了のため）

In [8]:
# ステップ3: 昇格判定
diff = challenger_acc - champion_acc
print(f"\n差分: {diff:+.4f}（チャレンジャー - チャンピオン）")

# 同スコアでもチャレンジャーを優先する場合は >= を使用
# より厳密な評価では統計的有意差を確認することを推奨
if challenger_acc >= champion_acc:
    print("\nChallenger wins! Promoting...")

    # @champion を新バージョン（チャレンジャー）に更新
    client.set_registered_model_alias(MODEL_NAME, "champion", challenger_mv.version)

    # 旧チャンピオンを @archived に移行（ロールバック用に保持）
    client.set_registered_model_alias(MODEL_NAME, "archived", champion_mv.version)

    # @challenger エイリアスを削除（評価フェーズ完了）
    client.delete_registered_model_alias(MODEL_NAME, "challenger")

    print(f"  @champion -> v{challenger_mv.version} ({challenger_type})")
    print(f"  @archived -> v{champion_mv.version} ({champion_type}, 旧チャンピオン)")
    print(f"  @challenger エイリアスを削除しました")
else:
    print("\nChampion holds position.")
    print(f"  @champion は v{champion_mv.version} のままです")


差分: +0.0000（チャレンジャー - チャンピオン）

Challenger wins! Promoting...
  @champion -> v2 (GradientBoosting)
  @archived -> v1 (RandomForest, 旧チャンピオン)
  @challenger エイリアスを削除しました


### 昇格後の検証

`get_model_version_by_alias()` を使って、エイリアスが正しく更新されたことをアサーションで確認します。これは本番環境での昇格スクリプトに組み込むことを推奨します。

In [9]:
# アサーション: @champion が正しく更新されたことを確認
new_champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
assert new_champion.version == challenger_mv.version, (
    f"昇格失敗！@champion は v{new_champion.version} ですが、"
    f"v{challenger_mv.version} になるはずです"
)

# アサーション: @archived に旧チャンピオンが保存されていることを確認
archived = client.get_model_version_by_alias(MODEL_NAME, "archived")
assert archived.version == champion_mv.version, (
    f"アーカイブ失敗！@archived は v{archived.version} ですが、"
    f"v{champion_mv.version} になるはずです"
)

print("=== 昇格後の検証 ===")
print(f"\n[OK] @champion は v{new_champion.version} になりました")
print(f"[OK] @archived には v{archived.version} が保存されています（ロールバック用）")

# 最終状態を一覧で確認
print("\n=== 最終エイリアス状態 ===")
all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
for mv in sorted(all_versions, key=lambda x: int(x.version)):
    print(f"バージョン v{mv.version}: aliases={mv.aliases}")

print("\n昇格が正常に完了しました！")

=== 昇格後の検証 ===

[OK] @champion は v2 になりました
[OK] @archived には v1 が保存されています（ロールバック用）

=== 最終エイリアス状態 ===
バージョン v1: aliases=['archived']
バージョン v2: aliases=['champion']

昇格が正常に完了しました！


## 6. ロールバック手順

新しいチャンピオンで問題が発生した場合、`@archived` に保存されている旧チャンピオンに素早くロールバックできます。

In [10]:
# ロールバック手順のデモ
# 実際の本番環境では問題を検知した際にこのコードを実行する

print("=== ロールバック手順（デモ） ===")

current_champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
archived_version = client.get_model_version_by_alias(MODEL_NAME, "archived")

print(f"\n現在の @champion: v{current_champion.version}")
print(f"ロールバック先 @archived: v{archived_version.version}")
print("\nロールバックを実行します...")

# @champion を旧バージョン（@archived）に戻す
client.set_registered_model_alias(MODEL_NAME, "champion", archived_version.version)

# @archived エイリアスを削除（ロールバック完了後はクリーンアップ）
client.delete_registered_model_alias(MODEL_NAME, "archived")

print(f"  @champion -> v{archived_version.version} に戻しました")
print(f"  @archived エイリアスを削除しました")

# ロールバック確認
rolled_back = client.get_model_version_by_alias(MODEL_NAME, "champion")
assert rolled_back.version == archived_version.version, "ロールバック失敗！"
print(f"\n[OK] ロールバック完了。@champion は v{rolled_back.version} です")

# 最終状態確認
print("\n=== ロールバック後のエイリアス状態 ===")
all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
for mv in sorted(all_versions, key=lambda x: int(x.version)):
    print(f"バージョン v{mv.version}: aliases={mv.aliases}")

=== ロールバック手順（デモ） ===

現在の @champion: v2
ロールバック先 @archived: v1

ロールバックを実行します...
  @champion -> v1 に戻しました
  @archived エイリアスを削除しました

[OK] ロールバック完了。@champion は v1 です

=== ロールバック後のエイリアス状態 ===
バージョン v1: aliases=['champion']
バージョン v2: aliases=[]


## 7. エイリアスAPI リファレンス

本ノートブックで使用したエイリアス関連APIをまとめます。

| API | 説明 |
|:----|:----|
| `client.set_registered_model_alias(name, alias, version)` | エイリアスを付与・更新する |
| `client.get_model_version_by_alias(name, alias)` | エイリアスでモデルバージョン情報を取得する |
| `client.delete_registered_model_alias(name, alias)` | エイリアスを削除する |
| `client.search_model_versions(filter_string)` | 条件でモデルバージョンを検索する |
| `client.get_run(run_id)` | Run情報（メトリクス・パラメータ）を取得する |
| `mlflow.pyfunc.load_model("models:/Name@alias")` | エイリアスURIでモデルをロードする |

### バージョン要件

```python
# エイリアスAPIはMLflow 2.8.0以降で利用可能
import mlflow
assert mlflow.__version__ >= "2.8.0", f"MLflow >= 2.8.0 が必要です（現在: {mlflow.__version__}）"
```

### エイリアスURIの形式

```
models://{モデル名}@{エイリアス名}

例:
  models:/IrisClassifier@champion    # チャンピオンモデル
  models:/IrisClassifier@challenger  # チャレンジャーモデル
  models:/IrisClassifier@archived    # アーカイブ済みモデル
```

> **参考**: バージョン番号でも同様にロードできます（例: `models:/IrisClassifier/1`）。ただし、エイリアスを使うことでコードを変更せずにモデルを切り替えられます。

---

## まとめ

このノートブックでは、MLflow のエイリアスAPIを使ったチャンピオン/チャレンジャーモデル管理を学びました。

### 学んだこと

| 学習項目 | 使用したAPI |
|:--------|:----------|
| エイリアスの付与 | `MlflowClient.set_registered_model_alias()` |
| エイリアスURIでのロード | `mlflow.pyfunc.load_model("models:/Name@alias")` |
| メトリクス比較 | `search_model_versions()` + `get_run()` |
| チャンピオン昇格 | エイリアスの付け替え |
| ロールバック | `@archived` エイリアスからの復元 |
| 昇格検証 | `get_model_version_by_alias()` + `assert` |

### ポイント

- **エイリアスはバージョン番号の抽象化**: アプリケーションのコードを変更せずにモデルを切り替えられる
- **`@archived` によるロールバック保証**: 昇格前に旧チャンピオンを保存しておくことで、問題発生時に即時ロールバックできる
- **アサーションによる検証**: 自動化スクリプトではエイリアス更新後に必ず検証を行うことを推奨
- **MLflow 2.8.0 以降が必要**: 旧バージョンの `stages`（Staging/Production）は非推奨。エイリアスへの移行を推奨

### 次のステップ

次のノートブックでは、`@champion` エイリアスを使って本番環境に FastAPI Webアプリケーションとしてデプロイする方法を学びます。

---

**次へ**: [06_deployment - FastAPIによるモデルデプロイ](../06_deployment/deployment.ipynb)